# Advanced Problems with Solutions: Creating Sets in Python

This notebook contains advanced, executable practice on **creating Python sets**.

Topics include:

- Set literals, `set()`, comprehensions, and unpacking
- Hashability and `frozenset`
- Dictionaries and generator inputs
- Normalization before deduplication
- Ordered deduplication
- Nested and JSON-like data
- Defensive validation
- Streaming algorithms
- Performance-aware construction
- Real-world parsing and indexing

> Sets are unordered collections of unique, hashable elements. Never rely on their display or iteration order.

## Setup

In [1]:
from collections.abc import Callable, Hashable, Iterable, Iterator
from dataclasses import dataclass
from itertools import chain
from pathlib import PurePath
from timeit import timeit
from typing import Any, TypeVar
from urllib.parse import urlsplit, urlunsplit
import re

T = TypeVar("T")
K = TypeVar("K", bound=Hashable)

## Construction Reference

In [2]:
known_values = {"red", "green", "blue"}
empty_set = set()
converted = set(range(5))
transformed = {n * n for n in range(-4, 5)}
combined = {*known_values, *converted}

assert empty_set == set()
assert converted == {0, 1, 2, 3, 4}
assert transformed == {0, 1, 4, 9, 16}

print("known_values:", known_values)
print("converted:", converted)
print("transformed:", transformed)
print("combined:", combined)

known_values: {'blue', 'red', 'green'}
converted: {0, 1, 2, 3, 4}
transformed: {0, 1, 4, 9, 16}
combined: {0, 1, 2, 'blue', 3, 4, 'red', 'green'}


### Best Practices

- Use `set()` for an empty set; `{}` is an empty dictionary.
- Prefer a set comprehension over `set([...])` when transforming data.
- Normalize values before deduplicating them.
- Never depend on set order.
- Verify hashability for nested or user-provided values.
- Use `frozenset` for immutable nested sets.
- Document whether a function consumes a one-shot iterable.

# Problem 1 — Diagnose Set Expressions

Classify each expression as a set, another type, or an exception.

In [3]:
factories = {
    "empty braces": lambda: {},
    "empty set": lambda: set(),
    "duplicates": lambda: {1, 2, 2, 3},
    "string conversion": lambda: set("mississippi"),
    "list elements": lambda: {[1, 2], [3, 4]},
    "empty tuple element": lambda: {()},
    "frozen nested sets": lambda: {
        frozenset({1, 2}),
        frozenset({2, 3}),
    },
}

diagnosis = {}

for label, factory in factories.items():
    try:
        value = factory()
        diagnosis[label] = (
            type(value).__name__,
            value,
        )
    except Exception as exc:
        diagnosis[label] = (
            type(exc).__name__,
            str(exc),
        )

for label, result in diagnosis.items():
    print(f"{label:20} -> {result}")

empty braces         -> ('dict', {})
empty set            -> ('set', set())
duplicates           -> ('set', {1, 2, 3})
string conversion    -> ('set', {'m', 'p', 'i', 's'})
list elements        -> ('TypeError', "unhashable type: 'list'")
empty tuple element  -> ('set', {()})
frozen nested sets   -> ('set', {frozenset({2, 3}), frozenset({1, 2})})


## Solution 1

`{}` is a dictionary. Lists are unhashable. Tuples and `frozenset` objects can be set elements when their contents are hashable.

In [4]:
assert type(factories["empty braces"]()) is dict
assert factories["empty set"]() == set()
assert factories["duplicates"]() == {1, 2, 3}
assert factories["string conversion"]() == set("misp")
assert factories["empty tuple element"]() == {()}
assert len(factories["frozen nested sets"]()) == 2

# Problem 2 — Normalize Noisy Tags

Create a normalized tag set:

- Trim whitespace
- Lowercase
- Collapse internal whitespace
- Replace spaces with hyphens
- Ignore empty results

In [5]:
raw_tags = [
    " Python ",
    "DATA SCIENCE",
    "",
    "python",
    "Data   Science",
    "  Machine Learning  ",
    "machine learning",
    "   ",
]

## Solution 2

In [6]:
def normalize_tag(tag):
    return "-".join(tag.strip().casefold().split())


normalized_tags = {
    normalized
    for tag in raw_tags
    if (normalized := normalize_tag(tag))
}

assert normalized_tags == {
    "python",
    "data-science",
    "machine-learning",
}

print(normalized_tags)

{'data-science', 'python', 'machine-learning'}


# Problem 3 — Convert Nested Lists into Hashable Coordinates

Keep only valid two-item numeric coordinates and convert them to tuples.

In [7]:
raw_coordinates = [
    [10, 20],
    [10, 20],
    [5.5, 9],
    [1],
    ["x", 2],
    [0, 0],
    [5.5, 9],
]

## Solution 3

In [8]:
def is_number(value):
    return (
        isinstance(value, (int, float))
        and not isinstance(value, bool)
    )


coordinates = set()

for item in raw_coordinates:
    if len(item) != 2:
        continue

    x, y = item

    if is_number(x) and is_number(y):
        coordinates.add((x, y))

assert coordinates == {
    (10, 20),
    (5.5, 9),
    (0, 0),
}

print(coordinates)

{(0, 0), (5.5, 9), (10, 20)}


# Problem 4 — Construct Sets from a Dictionary

Create sets of keys, values, items, and keys whose value is zero.

In [9]:
stock = {
    "keyboard": 12,
    "mouse": 0,
    "monitor": 7,
    "webcam": 0,
    "dock": 7,
}

## Solution 4

In [10]:
products = set(stock)
quantities = set(stock.values())
inventory_pairs = set(stock.items())
out_of_stock = {
    product
    for product, quantity in stock.items()
    if quantity == 0
}

assert products == {
    "keyboard",
    "mouse",
    "monitor",
    "webcam",
    "dock",
}
assert quantities == {0, 7, 12}
assert out_of_stock == {"mouse", "webcam"}

print("products:", products)
print("quantities:", quantities)
print("pairs:", inventory_pairs)
print("out_of_stock:", out_of_stock)

products: {'monitor', 'webcam', 'mouse', 'keyboard', 'dock'}
quantities: {0, 12, 7}
pairs: {('monitor', 7), ('keyboard', 12), ('webcam', 0), ('mouse', 0), ('dock', 7)}
out_of_stock: {'mouse', 'webcam'}


# Problem 5 — One-Shot Generators

Convert a generator to a set twice and explain the result.

## Solution 5

In [11]:
even_squares = (
    n * n
    for n in range(10)
    if n % 2 == 0
)

first = set(even_squares)
second = set(even_squares)

assert first == {0, 4, 16, 36, 64}
assert second == set()

print("first:", first)
print("second:", second)

first: {0, 64, 4, 36, 16}
second: set()


# Problem 6 — Deduplicate with and without Order

Return both an unordered unique set and a first-seen-order list.

In [12]:
event_ids = [
    "E3",
    "E1",
    "E3",
    "E2",
    "E1",
    "E4",
    "E2",
]

## Solution 6

In [13]:
unique_ids = set(event_ids)
ordered_unique_ids = list(dict.fromkeys(event_ids))

assert unique_ids == {"E1", "E2", "E3", "E4"}
assert ordered_unique_ids == ["E3", "E1", "E2", "E4"]

print("set:", unique_ids)
print("ordered:", ordered_unique_ids)

set: {'E1', 'E2', 'E3', 'E4'}
ordered: ['E3', 'E1', 'E2', 'E4']


# Problem 7 — Flatten Multiple Iterables into a Set

Solve with unpacking, `chain`, and a nested comprehension.

In [14]:
batch_a = [1, 2, 3, 3]
batch_b = (3, 4, 5)
batch_c = range(5, 9)

## Solution 7

In [15]:
by_unpacking = {*batch_a, *batch_b, *batch_c}
by_chain = set(chain(batch_a, batch_b, batch_c))
by_comprehension = {
    value
    for batch in (batch_a, batch_b, batch_c)
    for value in batch
}

expected = set(range(1, 9))

assert by_unpacking == expected
assert by_chain == expected
assert by_comprehension == expected

dynamic_batches = [batch_a, batch_b, batch_c]
dynamic_result = set(
    chain.from_iterable(dynamic_batches)
)
assert dynamic_result == expected

# Problem 8 — Create a Set of Sets

Treat each response as an unordered feature combination and remove duplicates.

In [16]:
responses = [
    ["search", "export"],
    ["export", "search"],
    ["alerts"],
    ["search", "alerts"],
    ["alerts", "search"],
]

## Solution 8

In [17]:
unique_combinations = {
    frozenset(response)
    for response in responses
}

assert unique_combinations == {
    frozenset({"search", "export"}),
    frozenset({"alerts"}),
    frozenset({"search", "alerts"}),
}

print(unique_combinations)

{frozenset({'export', 'search'}), frozenset({'alerts', 'search'}), frozenset({'alerts'})}


# Problem 9 — Canonicalize People before Deduplication

Normalize names and email addresses, then create a set of canonical tuples.

In [18]:
people = [
    {
        "name": "  ada   lovelace ",
        "email": "ADA@EXAMPLE.COM",
    },
    {
        "name": "Ada Lovelace",
        "email": "ada@example.com",
    },
    {
        "name": "grace hopper",
        "email": "Grace.Hopper@Example.com ",
    },
    {
        "name": "",
        "email": "missing@example.com",
    },
    {
        "name": "Alan Turing",
        "email": "  ",
    },
]

## Solution 9

In [19]:
def canonical_name(value):
    return " ".join(value.split()).title()


def canonical_email(value):
    return value.strip().casefold()


canonical_people = set()

for record in people:
    name = canonical_name(record["name"])
    email = canonical_email(record["email"])

    if name and email:
        canonical_people.add((name, email))

assert canonical_people == {
    ("Ada Lovelace", "ada@example.com"),
    ("Grace Hopper", "grace.hopper@example.com"),
}

print(canonical_people)

{('Ada Lovelace', 'ada@example.com'), ('Grace Hopper', 'grace.hopper@example.com')}


# Problem 10 — Alphabet Coverage

Create a set of lowercase ASCII letters used by a string and return a score from 0 to 1.

## Solution 10

In [20]:
ASCII_LETTERS = frozenset(
    "abcdefghijklmnopqrstuvwxyz"
)


def used_ascii_letters(text):
    return set(text.casefold()) & ASCII_LETTERS


def alphabet_coverage(text):
    return (
        len(used_ascii_letters(text))
        / len(ASCII_LETTERS)
    )


assert alphabet_coverage("baa baa") == 2 / 26
assert alphabet_coverage(
    "The quick brown fox jumps over the lazy dog"
) == 1.0
assert alphabet_coverage("123 !!!") == 0.0

print(used_ascii_letters("Set Theory!"))

{'e', 's', 'o', 'h', 't', 'r', 'y'}


# Problem 11 — Defensive Set Conversion

Raise a helpful error identifying the first unhashable value and its index.

## Solution 11

In [21]:
def to_hashable_set(iterable):
    result = set()

    for index, element in enumerate(iterable):
        try:
            hash(element)
        except TypeError as exc:
            raise TypeError(
                f"Element at index {index} is unhashable: "
                f"{element!r}"
            ) from exc

        result.add(element)

    return result


assert to_hashable_set(
    value for value in [1, 2, 2, 3]
) == {1, 2, 3}

try:
    to_hashable_set([1, 2, ["bad"], 4])
except TypeError as exc:
    print(exc)
else:
    raise AssertionError("Expected TypeError")

Element at index 2 is unhashable: ['bad']


# Problem 12 — Recursively Freeze Nested Data

Convert common mutable containers to hashable equivalents.

## Solution 12

In [22]:
def freeze(value):
    if isinstance(value, dict):
        frozen_items = (
            (freeze(key), freeze(item_value))
            for key, item_value in value.items()
        )
        return tuple(
            sorted(frozen_items, key=repr)
        )

    if isinstance(value, list):
        return tuple(
            freeze(item)
            for item in value
        )

    if isinstance(value, set):
        return frozenset(
            freeze(item)
            for item in value
        )

    if isinstance(value, tuple):
        return tuple(
            freeze(item)
            for item in value
        )

    hash(value)
    return value


nested_records = [
    {
        "id": 1,
        "roles": ["admin", "editor"],
        "flags": {"active", "beta"},
    },
    {
        "flags": {"beta", "active"},
        "roles": ["admin", "editor"],
        "id": 1,
    },
    {
        "id": 2,
        "roles": ["viewer"],
        "flags": {"active"},
    },
]

unique_nested_records = {
    freeze(record)
    for record in nested_records
}

assert len(unique_nested_records) == 2
print(unique_nested_records)

{(('flags', frozenset({'active', 'beta'})), ('id', 1), ('roles', ('admin', 'editor'))), (('flags', frozenset({'active'})), ('id', 2), ('roles', ('viewer',)))}


# Problem 13 — Unique Words with Normalization

Keep apostrophes inside words, lowercase tokens, and ignore words shorter than three characters.

In [23]:
sample_text = (
    "Python's sets are fast. Sets remove duplicates; "
    "PYTHON'S syntax is concise, and sets support "
    "membership tests."
)

## Solution 13

In [24]:
WORD_PATTERN = re.compile(
    r"[a-z]+(?:'[a-z]+)?"
)


def unique_words(text, min_length=3):
    return {
        word
        for word in WORD_PATTERN.findall(
            text.casefold()
        )
        if len(word) >= min_length
    }


words = unique_words(sample_text)

assert "python's" in words
assert "sets" in words
assert "is" not in words
assert words == set(words)

print(words)

{'tests', 'support', 'duplicates', 'membership', "python's", 'fast', 'remove', 'are', 'syntax', 'and', 'concise', 'sets'}


# Problem 14 — Hashable Domain Objects

Create immutable objects that can safely be set elements.

## Solution 14

In [25]:
@dataclass(frozen=True, slots=True)
class User:
    user_id: int
    username: str


users = {
    User(1, "ada"),
    User(2, "grace"),
    User(1, "ada"),
}

assert len(users) == 2
print(users)

{User(user_id=1, username='ada'), User(user_id=2, username='grace')}


# Problem 15 — Streaming Duplicate Detection

Yield each duplicate only once, at the moment it first becomes a duplicate.

## Solution 15

In [26]:
def first_duplicate_occurrences(items):
    seen = set()
    reported = set()

    for item in items:
        if item in seen and item not in reported:
            reported.add(item)
            yield item
        else:
            seen.add(item)


duplicates = list(
    first_duplicate_occurrences(
        [1, 2, 1, 3, 2, 2, 4]
    )
)

assert duplicates == [1, 2]
print(duplicates)

[1, 2]


# Problem 16 — Deduplicate by a Custom Key

Keep the first item for each case-insensitive email address.

## Solution 16

In [27]:
def unique_by(iterable, key):
    seen_keys = set()
    result = []

    for item in iterable:
        marker = key(item)

        if marker not in seen_keys:
            seen_keys.add(marker)
            result.append(item)

    return result


emails = [
    "Ada@Example.com",
    "ada@example.com",
    "Grace@Example.com",
    "GRACE@example.com",
]

deduplicated = unique_by(
    emails,
    key=str.casefold,
)

assert deduplicated == [
    "Ada@Example.com",
    "Grace@Example.com",
]

print(deduplicated)

['Ada@Example.com', 'Grace@Example.com']


# Problem 17 — Distinct Matrix Values with Shape Validation

Collect distinct values while rejecting ragged input.

## Solution 17

In [28]:
def distinct_matrix_values(rows):
    distinct = set()
    expected_width = None

    for row_index, row in enumerate(rows):
        row_values = tuple(row)

        if expected_width is None:
            expected_width = len(row_values)
        elif len(row_values) != expected_width:
            raise ValueError(
                f"Row {row_index} has width "
                f"{len(row_values)}; expected "
                f"{expected_width}"
            )

        distinct.update(row_values)

    return distinct


matrix = (
    range(start, start + 3)
    for start in [0, 2, 4]
)

assert distinct_matrix_values(matrix) == {
    0, 1, 2, 3, 4, 5, 6
}

try:
    distinct_matrix_values([[1, 2], [3]])
except ValueError as exc:
    print(exc)
else:
    raise AssertionError("Expected ValueError")

Row 1 has width 1; expected 2


# Problem 18 — Compare Direct and Indirect Construction

Compare `set([f(x) for x in data])` with `{f(x) for x in data}`.

## Solution 18

In [29]:
benchmark_data = list(range(10_000))


def transform(value):
    return (value * value) % 997


def list_then_set():
    return set([
        transform(value)
        for value in benchmark_data
    ])


def direct_comprehension():
    return {
        transform(value)
        for value in benchmark_data
    }


assert list_then_set() == direct_comprehension()

list_time = timeit(
    list_then_set,
    number=25,
)
direct_time = timeit(
    direct_comprehension,
    number=25,
)

print(f"list then set: {list_time:.4f}s")
print(f"direct set:    {direct_time:.4f}s")

list then set: 0.0720s
direct set:    0.0603s


# Problem 19 — Active-User Membership Index

Build a validated set of active integer user IDs.

In [30]:
user_records = [
    {"id": 101, "active": True},
    {"id": 102, "active": False},
    {"id": 103, "active": True},
    {"id": 101, "active": True},
]

## Solution 19

In [31]:
def active_user_ids(records):
    result = set()

    for index, record in enumerate(records):
        user_id = record.get("id")

        if (
            not isinstance(user_id, int)
            or isinstance(user_id, bool)
        ):
            raise TypeError(
                f"Invalid ID in record {index}: "
                f"{user_id!r}"
            )

        if record.get("active") is True:
            result.add(user_id)

    return result


active_ids = active_user_ids(user_records)

assert active_ids == {101, 103}
assert 101 in active_ids
assert 102 not in active_ids

# Problem 20 — Partition a Universe

Create accepted and rejected sets from a predicate, then prove they are disjoint and complete.

## Solution 20

In [32]:
def partition_set(universe, predicate):
    universe_set = set(universe)
    accepted = {
        item
        for item in universe_set
        if predicate(item)
    }
    rejected = universe_set - accepted
    return accepted, rejected


numbers = range(-10, 11)
nonnegative, negative = partition_set(
    numbers,
    lambda value: value >= 0,
)

assert nonnegative.isdisjoint(negative)
assert nonnegative | negative == set(numbers)
assert nonnegative == set(range(0, 11))
assert negative == set(range(-10, 0))

# Problem 21 — Parse and Deduplicate Network Endpoints

Canonicalize `host:port` strings and validate the port range.

In [33]:
raw_endpoints = [
    " API.EXAMPLE.COM:443 ",
    "api.example.com:443",
    "db.example.com:5432",
    "CACHE.EXAMPLE.COM:6379",
]

## Solution 21

In [34]:
def parse_endpoint(value):
    host_part, separator, port_part = (
        value.strip().rpartition(":")
    )

    if not separator or not host_part:
        raise ValueError(
            f"Invalid endpoint: {value!r}"
        )

    host = host_part.strip().casefold()

    try:
        port = int(port_part)
    except ValueError as exc:
        raise ValueError(
            f"Invalid port: {value!r}"
        ) from exc

    if not 1 <= port <= 65_535:
        raise ValueError(
            f"Port out of range: {value!r}"
        )

    return host, port


endpoints = {
    parse_endpoint(value)
    for value in raw_endpoints
}

assert endpoints == {
    ("api.example.com", 443),
    ("db.example.com", 5432),
    ("cache.example.com", 6379),
}

print(endpoints)

{('cache.example.com', 6379), ('db.example.com', 5432), ('api.example.com', 443)}


# Problem 22 — Distinct Sliding Windows

Return every distinct window of length `k` as a tuple.

## Solution 22

In [35]:
def distinct_windows(sequence, k):
    if k <= 0:
        raise ValueError("k must be positive")

    values = tuple(sequence)

    if k > len(values):
        return set()

    return {
        values[start:start + k]
        for start in range(
            len(values) - k + 1
        )
    }


windows = distinct_windows(
    [1, 2, 1, 2, 3],
    3,
)

assert windows == {
    (1, 2, 1),
    (2, 1, 2),
    (1, 2, 3),
}
assert distinct_windows([1, 2], 3) == set()

print(windows)

{(1, 2, 1), (2, 1, 2), (1, 2, 3)}


# Problem 23 — Set Equality Edge Cases

## Solution 23

In [36]:
# Empty braces are a dictionary.
assert isinstance({}, dict)

# Equal numeric values collapse.
numeric_set = {
    False,
    0,
    0.0,
    True,
    1,
    1.0,
}
assert numeric_set == {False, True}

# Distinct NaN objects can coexist.
nan_a = float("nan")
nan_b = float("nan")
nan_values = {nan_a, nan_b}
assert len(nan_values) == 2

# Dictionaries produce keys.
mapping = {"a": 1, "b": 2}
assert set(mapping) == {"a", "b"}

# Strings produce characters.
assert set("aba") == {"a", "b"}

# Tuple hashability depends on contents.
hash((1, 2))

try:
    hash((1, [2]))
except TypeError:
    pass
else:
    raise AssertionError("Expected TypeError")

print("numeric_set:", numeric_set)
print("NaN count:", len(nan_values))

numeric_set: {False, True}
NaN count: 2


# Problem 24 — Property-Style Tests

Test normalization, idempotence, duplicate-insensitivity, and order independence.

## Solution 24

In [37]:
def normalized_word_set(words):
    return {
        word.strip().casefold()
        for word in words
        if word.strip()
    }


test_words = [
    " Alpha ",
    "BETA",
    "alpha",
    "",
    " beta ",
]

result = normalized_word_set(test_words)

assert result == {"alpha", "beta"}
assert normalized_word_set(result) == result
assert normalized_word_set(
    test_words + test_words
) == result
assert normalized_word_set(
    reversed(test_words)
) == result

# Problem 25 — Capstone: Inverted Search Index

Map each normalized token to the set of document IDs containing it.

In [38]:
documents = [
    (
        101,
        "Python sets provide fast membership tests.",
    ),
    (
        102,
        "Sets remove duplicate values in Python.",
    ),
    (
        103,
        "Membership testing with a set is concise.",
    ),
    (
        101,
        "Python sets provide fast membership tests.",
    ),
]

## Solution 25

In [39]:
def build_inverted_index(
    records,
    min_token_length=3,
):
    if min_token_length <= 0:
        raise ValueError(
            "min_token_length must be positive"
        )

    index = {}

    for document_id, text in records:
        hash(document_id)

        tokens = {
            token
            for token in re.findall(
                r"[a-z0-9]+",
                text.casefold(),
            )
            if len(token) >= min_token_length
        }

        for token in tokens:
            index.setdefault(
                token,
                set(),
            ).add(document_id)

    return index


index = build_inverted_index(documents)

assert index["python"] == {101, 102}
assert index["sets"] == {101, 102}
assert index["membership"] == {101, 103}
assert index["duplicate"] == {102}

for token in sorted(index):
    print(
        f"{token:12} -> "
        f"{sorted(index[token])}"
    )

concise      -> [103]
duplicate    -> [102]
fast         -> [101]
membership   -> [101, 103]
provide      -> [101]
python       -> [101, 102]
remove       -> [102]
set          -> [103]
sets         -> [101, 102]
testing      -> [103]
tests        -> [101]
values       -> [102]
with         -> [103]


# Additional Examples and Challenge Solutions

## A. Unique File Extensions

In [40]:
paths = [
    "report.CSV",
    "archive.tar.gz",
    "README",
    "image.PNG",
    "data.csv",
]

extensions = {
    PurePath(path).suffix.casefold()
    for path in paths
    if PurePath(path).suffix
}

assert extensions == {
    ".csv",
    ".gz",
    ".png",
}

## B. Canonical Phone Numbers

In [41]:
phone_numbers = [
    "+1 (555) 123-4567",
    "15551234567",
    "+1-555-123-4567",
    "(555) 000-0000",
]

canonical_phones = {
    "".join(
        character
        for character in number
        if character.isdigit()
    )
    for number in phone_numbers
}

assert canonical_phones == {
    "15551234567",
    "5550000000",
}

## C. Canonical Undirected Graph Edges

In [42]:
adjacency = {
    "A": ["B", "C"],
    "B": ["A", "C"],
    "C": ["A", "B"],
}

undirected_edges = {
    frozenset((source, target))
    for source, targets in adjacency.items()
    for target in targets
    if source != target
}

assert undirected_edges == {
    frozenset({"A", "B"}),
    frozenset({"A", "C"}),
    frozenset({"B", "C"}),
}

## D. Distinct Pairs Summing to a Target

In [43]:
def pairs_with_sum(values, target):
    seen = set()
    pairs = set()

    for value in values:
        complement = target - value

        if complement in seen:
            pairs.add(
                tuple(sorted(
                    (value, complement)
                ))
            )

        seen.add(value)

    return pairs


assert pairs_with_sum(
    [1, 2, 3, 4, 5, 3, 2],
    6,
) == {
    (1, 5),
    (2, 4),
    (3, 3),
}

## E. Unique Nested Database Rows

In [44]:
database_rows = [
    {
        "id": 1,
        "metadata": {"tags": ["a", "b"]},
    },
    {
        "metadata": {"tags": ["a", "b"]},
        "id": 1,
    },
    {
        "id": 2,
        "metadata": {"tags": ["b"]},
    },
]

unique_rows = {
    freeze(row)
    for row in database_rows
}

assert len(unique_rows) == 2

## F. Distinct Unicode Letters

In [45]:
multilingual_text = (
    "Hello, Καλημέρα, Здравствуйте, 你好!"
)

unicode_letters = {
    character.casefold()
    for character in multilingual_text
    if character.isalpha()
}

assert "h" in unicode_letters
assert "κ" in unicode_letters
assert "з" in unicode_letters
assert "你" in unicode_letters

## G. Effective Permissions from Roles

In [46]:
role_permissions = {
    "viewer": {"read"},
    "editor": {"read", "write"},
    "admin": {
        "read",
        "write",
        "delete",
        "manage-users",
    },
}

assigned_roles = [
    "viewer",
    "editor",
    "viewer",
]

effective_permissions = set().union(
    *(
        role_permissions[role]
        for role in assigned_roles
    )
)

assert effective_permissions == {
    "read",
    "write",
}

## H. Dates Shared by Several Streams

In [47]:
stream_a = {
    "2026-08-01",
    "2026-08-02",
    "2026-08-04",
}
stream_b = {
    "2026-08-02",
    "2026-08-03",
    "2026-08-04",
}
stream_c = {
    "2026-08-02",
    "2026-08-04",
    "2026-08-05",
}

shared_dates = set.intersection(
    stream_a,
    stream_b,
    stream_c,
)

assert shared_dates == {
    "2026-08-02",
    "2026-08-04",
}

## I. Duplicate Composite Keys

In [48]:
rows = [
    {
        "customer_id": 1,
        "date": "2026-08-01",
        "amount": 10,
    },
    {
        "customer_id": 2,
        "date": "2026-08-01",
        "amount": 20,
    },
    {
        "customer_id": 1,
        "date": "2026-08-01",
        "amount": 30,
    },
]

composite_keys = [
    (
        row["customer_id"],
        row["date"],
    )
    for row in rows
]

duplicate_keys = set(
    first_duplicate_occurrences(
        composite_keys
    )
)

assert duplicate_keys == {
    (1, "2026-08-01")
}

## J. Canonical URLs

In [49]:
def canonical_url(url):
    parts = urlsplit(url.strip())
    scheme = parts.scheme.casefold()
    hostname = (
        parts.hostname or ""
    ).casefold()

    if not scheme or not hostname:
        raise ValueError(
            f"Absolute URL required: {url!r}"
        )

    port = parts.port
    default_port = (
        (scheme == "http" and port == 80)
        or (
            scheme == "https"
            and port == 443
        )
    )

    if port is None or default_port:
        host = hostname
    else:
        host = f"{hostname}:{port}"

    path = parts.path or "/"

    return urlunsplit(
        (
            scheme,
            host,
            path,
            parts.query,
            "",
        )
    )


urls = [
    "HTTPS://Example.COM:443/docs#intro",
    "https://example.com/docs",
    "https://example.com:8443/docs",
]

canonical_urls = {
    canonical_url(url)
    for url in urls
}

assert canonical_urls == {
    "https://example.com/docs",
    "https://example.com:8443/docs",
}

# Final Checklist

Before creating a set, ask:

- Are all elements hashable?
- Is unordered uniqueness appropriate?
- Must first-seen order be preserved?
- Should values be normalized first?
- Could the input be a one-shot iterator?
- Am I accidentally iterating a string into characters?
- Am I relying on display order?
- Should nested groups use `frozenset`?
- Can I avoid an intermediate list?
- Are equality edge cases such as `True == 1` relevant?

# Summary

```python
# Known elements
values = {1, 2, 3}

# Empty set
values = set()

# Convert an iterable
values = set(iterable)

# Transform and filter
values = {
    transform(item)
    for item in iterable
    if predicate(item)
}

# Combine dynamic iterables
values = set(
    chain.from_iterable(iterables)
)

# Immutable nested groups
groups = {
    frozenset(group)
    for group in group_source
}
```

Advanced set construction depends on correct rules for identity, normalization, hashability, ordering, and validation.